# 2 — Parallel CSV Batch Product Resolution

Use this notebook when a CSV contains many products.

Required columns:

```text
main_text
country_code
```

Optional columns:

```text
row_id
ean
retailer_name
language_code
```

Accepted uppercase/spaced aliases are normalized automatically. Every product receives an isolated artifact directory. Failures are captured per row and do not stop the complete batch.


In [ ]:
from __future__ import annotations

import importlib
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "docker-compose.yml").is_file() and (candidate / "src" / "product_evidence_harness").is_dir():
            return candidate
    raise RuntimeError("Could not locate the web_search_tool repository root")

PROJECT_ROOT = find_project_root()
for required in (str(PROJECT_ROOT), str(PROJECT_ROOT / "src")):
    if required not in sys.path:
        sys.path.insert(0, required)
importlib.invalidate_caches()

from product_evidence_harness.notebook_runtime import DEFAULT_FEATURE_SET, ensure_platform_ready
from product_evidence_harness.batch_notebook_runtime import (
    load_batch_csv,
    normalize_batch_input,
    recommended_batch_parallelism,
    run_batch_products,
)

health, platform_recovery = ensure_platform_ready(
    PROJECT_ROOT,
    auto_recover=True,
    clean_build=True,
)
recommended_parallel = recommended_batch_parallelism()
readiness_df = pd.DataFrame([{
    "platform_status": health.get("status"),
    "runtime_contract": health.get("runtime_contract_version"),
    "manufacturer_first_primary_url": health.get("manufacturer_first_primary_url"),
    "business_judgement_review_artifact": health.get("business_judgement_review_artifact"),
    "recommended_parallel_products": recommended_parallel,
    "agent_workers_environment": os.getenv("AGENT_WORKERS", "2"),
    "browser_contexts_environment": os.getenv("BROWSER_MAX_CONTEXTS", "3"),
    "auto_recovery_attempted": platform_recovery.attempted,
}])
display(readiness_df)


## Configure the batch

The default parallelism is bounded by `AGENT_WORKERS` and `BROWSER_MAX_CONTEXTS`. Increase those `.env` values only after load testing and then rebuild the services.


In [ ]:
CSV_PATH = PROJECT_ROOT / "examples" / "batch_products.example.csv"
FEATURE_SET = DEFAULT_FEATURE_SET
MAX_PARALLEL_PRODUCTS = recommended_parallel
RUN_BATCH = False

print(f"CSV: {CSV_PATH}")
print(f"Parallel products: {MAX_PARALLEL_PRODUCTS}")


## Load and validate the CSV

EAN remains text. Blank mandatory fields and duplicate `row_id` values fail before any paid search.


In [ ]:
raw_batch_df = load_batch_csv(CSV_PATH)
normalized_batch_df = normalize_batch_input(raw_batch_df)
display(normalized_batch_df)
print(f"Validated rows: {len(normalized_batch_df)}")


## Execute the parallel batch

Set `RUN_BATCH = True` after reviewing the normalized input.


In [ ]:
if RUN_BATCH:
    batch_report = run_batch_products(
        normalized_batch_df,
        project_root=PROJECT_ROOT,
        feature_set=FEATURE_SET,
        max_parallel=MAX_PARALLEL_PRODUCTS,
        print_progress=True,
    )
    print(f"Batch completed: {batch_report.run_id}")
    print(f"Output directory: {batch_report.output_dir}")
else:
    print("Ready. Set RUN_BATCH = True to execute.")


# Batch result view


In [ ]:
if "batch_report" not in globals():
    raise RuntimeError("Run the batch execution cell first")

batch_summary_df = pd.DataFrame([{
    "run_id": batch_report.summary.get("run_id"),
    "input_rows": batch_report.summary.get("input_rows"),
    "max_parallel": batch_report.summary.get("max_parallel"),
    "successful_or_review_rows": batch_report.summary.get("successful_or_review_rows"),
    "failed_rows": batch_report.summary.get("failed_rows"),
    "elapsed_seconds": batch_report.summary.get("elapsed_seconds"),
    "throughput_products_per_minute": batch_report.summary.get("throughput_products_per_minute"),
    "mean_product_elapsed_seconds": batch_report.summary.get("mean_product_elapsed_seconds"),
    "p50_product_elapsed_seconds": batch_report.summary.get("p50_product_elapsed_seconds"),
    "p95_product_elapsed_seconds": batch_report.summary.get("p95_product_elapsed_seconds"),
    "total_serpapi_credits_used": batch_report.summary.get("total_serpapi_credits_used"),
    "output_dir": batch_report.summary.get("output_dir"),
}])
display(batch_summary_df)

result_columns = [
    "row_id",
    "main_text",
    "ean",
    "retailer_name",
    "country_code",
    "job_status",
    "primary_url",
    "primary_url_role",
    "manufacturer_url",
    "retailer_url",
    "selection_reason",
    "strictly_verified",
    "search_credits_used",
    "image_influenced_final_decision",
    "elapsed_seconds",
    "artifact_dir",
    "business_judgement_review_path",
    "error",
]
display(batch_report.results_df[result_columns])


## Throughput and outcome diagnostics


In [ ]:
status_counts = batch_report.results_df["job_status"].value_counts()
fig, ax = plt.subplots(figsize=(8, 4))
status_counts.plot(kind="bar", ax=ax)
ax.set_title("Batch outcomes")
ax.set_xlabel("Outcome")
ax.set_ylabel("Products")
plt.xticks(rotation=0)
plt.show()

elapsed = pd.to_numeric(batch_report.results_df["elapsed_seconds"], errors="coerce").dropna()
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(elapsed, bins=max(5, min(20, len(elapsed))))
ax.set_title("Per-product elapsed time")
ax.set_xlabel("Seconds")
ax.set_ylabel("Products")
plt.show()


## Failures and artifact index

Every successful or review-required row links to the complete product artifact and business judgment Markdown.


In [ ]:
display(batch_report.failures_df)
display(batch_report.artifact_index_df)

for name in (
    "batch_input_normalized.csv",
    "batch_results.csv",
    "batch_failures.csv",
    "batch_artifact_index.csv",
    "batch_run_summary.json",
):
    print(batch_report.output_dir / name)


## Throughput interpretation

`MAX_PARALLEL_PRODUCTS` controls product-level concurrency. Each product still retains its own bounded search, scrape and browser budgets.

A higher number is not automatically faster: browser contexts, agent workers, SerpAPI limits, LLM rate limits and page latency determine safe throughput. Use the batch summary's elapsed time, p50, p95 and products/minute to tune this empirically.
